# Exploratory Analysis of Bioactive Compounds Targeting EGFR

Pipeline de classification de bioactivite pour des composes ciblant EGFR (Epidermal Growth Factor Receptor, ChEMBL205).

**Auteur** : Hakim Ben Hamouda — INSAT Tunis

> Code reconstruit depuis l historique d execution IPython de la session du 27 juin 2026.
> Les sorties ne sont pas incluses : relancer le notebook pour les regenerer.


In [ ]:
# Cellule 1 — Imports et vérification
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from chembl_webresource_client.new_client import new_client

print("Toutes les bibliothèques sont chargées ✓")

In [ ]:
# Cellule 2 — Téléchargement des données depuis ChEMBL
activity = new_client.activity
results = activity.filter(
    target_chembl_id="CHEMBL205",  # Kinase EGFR — cible anticancer majeure
    standard_type="IC50",
    relation="=",
    assay_type="B"
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value",
    "standard_units",
    "pchembl_value"
])

df = pd.DataFrame.from_records(results)
print(f"Nombre de composés téléchargés : {len(df)}")
print(df.head())

In [ ]:
# Cellule 3 — Nettoyage des données
df = df[["molecule_chembl_id", "canonical_smiles", "standard_value", "pchembl_value"]].copy()
df = df.dropna()
df = df[df["standard_value"].astype(float) > 0]
df["standard_value"] = df["standard_value"].astype(float)
df["pchembl_value"] = df["pchembl_value"].astype(float)
df = df.drop_duplicates(subset="molecule_chembl_id")
df = df.reset_index(drop=True)

print(f"Après nettoyage : {len(df)} composés")
print(df.dtypes)

In [ ]:
# Cellule 4 — Classification actif / inactif
# Convention standard en drug discovery :
# pChEMBL >= 6  →  IC50 <= 1 µM  →  ACTIF
# pChEMBL < 6   →  IC50 > 1 µM   →  INACTIF

df["active"] = (df["pchembl_value"] >= 6).astype(int)

actifs = df["active"].sum()
inactifs = len(df) - actifs

print(f"Composés actifs   (pChEMBL ≥ 6) : {actifs}")
print(f"Composés inactifs (pChEMBL < 6) : {inactifs}")
print(f"\nDistribution : {actifs/len(df)*100:.1f}% actifs")

In [ ]:
# Cellule 5 — Distribution des pChEMBL values
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogramme
axes[0].hist(df["pchembl_value"], bins=30, color="#2ecc71", edgecolor="white", alpha=0.85)
axes[0].axvline(x=6, color="red", linestyle="--", linewidth=1.5, label="Seuil actif (pChEMBL=6)")
axes[0].set_xlabel("pChEMBL Value", fontsize=12)
axes[0].set_ylabel("Nombre de composés", fontsize=12)
axes[0].set_title("Distribution des pChEMBL Values\n(EGFR, ChEMBL205)", fontsize=13, fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pie chart actif/inactif
labels = ["Actifs", "Inactifs"]
sizes = [637, 183]
colors = ["#2ecc71", "#e74c3c"]
axes[1].pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Répartition Actifs / Inactifs", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("figure1_distribution.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Figure sauvegardée ✓")

In [ ]:
# Cellule 6 — Calcul des descripteurs moléculaires avec RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def calculer_descripteurs(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW":    Descriptors.MolWt(mol),           # Masse moléculaire
        "LogP":  Descriptors.MolLogP(mol),          # Lipophilicité
        "HBD":   rdMolDescriptors.CalcNumHBD(mol),  # H-bond donneurs
        "HBA":   rdMolDescriptors.CalcNumHBA(mol),  # H-bond accepteurs
        "TPSA":  Descriptors.TPSA(mol),             # Surface polaire
        "RotBonds": rdMolDescriptors.CalcNumRotatableBonds(mol)
    }

descripteurs = df["canonical_smiles"].apply(calculer_descripteurs)
df_desc = pd.DataFrame(descripteurs.tolist())
df_final = pd.concat([df, df_desc], axis=1).dropna()
df_final = df_final.reset_index(drop=True)

print(f"Composés avec descripteurs valides : {len(df_final)}")
print(df_final[["MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds"]].describe().round(2))

In [ ]:
# Cellule 7 — Analyse des descripteurs + Règle de Lipinski
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

descripteurs = ["MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds"]
limites_lipinski = {"MW": 500, "LogP": 5, "HBD": 5, "HBA": 10, "TPSA": 140, "RotBonds": 10}
couleurs = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]

for i, (desc, color) in enumerate(zip(descripteurs, couleurs)):
    axes[i].hist(df_final[desc], bins=25, color=color, edgecolor="white", alpha=0.85)
    if desc in limites_lipinski:
        axes[i].axvline(x=limites_lipinski[desc], color="black",
                       linestyle="--", linewidth=1.5, label=f"Lipinski ({limites_lipinski[desc]})")
        axes[i].legend(fontsize=9)
    axes[i].set_xlabel(desc, fontsize=11)
    axes[i].set_ylabel("Nombre de composés", fontsize=10)
    axes[i].set_title(f"Distribution — {desc}", fontsize=11, fontweight="bold")
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Propriétés Physico-chimiques des Inhibiteurs EGFR\n(Règles de Lipinski)", 
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("figure2_lipinski.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Figure sauvegardée ✓")

In [ ]:
# Cellule 8 — Modèle de classification Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

# Features et target
X = df_final[["MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds"]]
y = df_final["active"]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entraînement
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)

print(f"AUC-ROC : {auc:.3f}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=["Inactif", "Actif"]))

In [ ]:
# Cellule 9 — Visualisation ROC + Matrice de confusion + Feature Importance
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color="#3498db", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[0].plot([0,1], [0,1], color="gray", linestyle="--", lw=1)
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#3498db")
axes[0].set_xlabel("False Positive Rate", fontsize=11)
axes[0].set_ylabel("True Positive Rate", fontsize=11)
axes[0].set_title("Courbe ROC", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Inactif","Actif"],
            yticklabels=["Inactif","Actif"])
axes[1].set_xlabel("Prédit", fontsize=11)
axes[1].set_ylabel("Réel", fontsize=11)
axes[1].set_title("Matrice de Confusion", fontsize=13, fontweight="bold")

# Feature Importance
importances = pd.Series(model.feature_importances_,
                        index=["MW","LogP","HBD","HBA","TPSA","RotBonds"])
importances.sort_values().plot(kind="barh", ax=axes[2], color="#2ecc71", edgecolor="white")
axes[2].set_xlabel("Importance", fontsize=11)
axes[2].set_title("Feature Importance\n(Random Forest)", fontsize=13, fontweight="bold")
axes[2].grid(True, alpha=0.3, axis="x")

plt.suptitle("Évaluation du Modèle — Classification Actif/Inactif EGFR",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("figure3_model.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Figure sauvegardée ✓")